In [66]:
# General imports
import pandas as pd
import numpy as np
import itertools
import math
from tqdm import tqdm

# Import custom LIF SNN implementation
from LIF_SNN_network import SNNLayer

# Set random seed for reproducability
np.random.seed(42)

**Test data**

In [67]:
spiketrains = pd.read_csv("Frame_test_spiketrains.csv")

test_dist_train = [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 
                   0, 0, 0, 0, 0, 0, 1, 0, 0, 0,
                   0, 0, 1, 0, 0, 0, 0, 0, 0, 0,
                   0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
                   0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

test_obj_req = [
    [0, 0, 0],  # 0
    [0, 0, 0],  # 1
    [0, 0, 0],  # 2
    [0, 0, 0],  # 3
    [0, 0, 0],  # 4
    [0, 0, 0],  # 5
    [0, 0, 0],  # 6
    [0, 0, 0],  # 7
    [0, 0, 0],  # 8
    [0, 0, 0],  # 9
    [0, 0, 0],  # 10
    [0, 0, 0],  # 11
    [0, 0, 0],  # 12
    [0, 0, 0],  # 13
    [0, 0, 0],  # 14
    [1, 0, 0],  # 15 - Object detected LEFT
    [1, 0, 0],  # 16
    [1, 0, 0],  # 17
    [1, 0, 0],  # 18
    [1, 0, 0],  # 19
    [1, 0, 0],  # 20
    [1, 0, 0],  # 21
    [1, 0, 0],  # 22
    [1, 0, 0],  # 23
    [1, 0, 0],  # 24
    [0, 1, 0],  # 25 - Object detected CENTRE
    [0, 1, 0],  # 26
    [0, 1, 0],  # 27
    [0, 1, 0],  # 28
    [0, 1, 0],  # 29
    [0, 1, 0],  # 30
    [0, 1, 0],  # 31
    [0, 0, 1],  # 32 - Object detected RIGHT
    [0, 0, 1],  # 33
    [0, 0, 1],  # 34
    [0, 0, 1],  # 35
    [0, 0, 1],  # 36 
    [0, 0, 1],  # 37
    [0, 0, 1],  # 38
    [0, 0, 1],  # 39
    [0, 1, 1],  # 40 - Object detected CENTRE again
    [0, 1, 0],  # 41
    [0, 1, 0],  # 42
    [0, 1, 0],  # 43
    [0, 1, 0],  # 44
    [0, 1, 0],  # 45
    [0, 1, 0],  # 46
    [0, 1, 0],  # 47
    [0, 1, 0],  # 48
    [0, 1, 0],  # 49
]

correct_outputs = []
for i in range(len(test_obj_req)):
    obj_l, obj_c, obj_r = test_obj_req[i]
    
    if obj_r:
        correct_outputs.append(2)      # object right → turn right
    elif obj_l:
        correct_outputs.append(1)      # object left → turn left
    elif obj_c or test_dist_train[i]:
        correct_outputs.append(1)      # object center or close → forward
    else:
        correct_outputs.append(1)      # nothing → forward/explore

input_spikes = []
for i in range(len(spiketrains)):
    row = spiketrains.iloc[i].tolist() + [test_dist_train[i]] + test_obj_req[i]
    input_spikes.append(row)

**Parameters**

In [68]:
# Input/Output size
n_inputs = 16
n_outputs = 3

# Training params
n_epochs = 50

# Neuron hyperparameters
decay_range = [0.25, 0.5, 0.75]
threshold_range = [1.0, 2.0, 4.0]
reset_range = [0.0]

# Synapse parameters
learning_rate_range = [0.0625, 0.125]
initial_weight_range = [None]
t_pre_range = [2, 4]
t_post_range = [2, 4]
tau_e_shift_range = [2, 3]
dw_pos_range = [0.0625, 0.125]
dw_neg_range = [0.0625, 0.125]
min_weight_range = [0, 0.03125]
max_weight_range = [1.0]
dopamine_correct_range = [0.5, 1, 2]
dopamine_wrong_range = [-0.5, -1, -2]

In [69]:
# Calculate total combinations and set up all configurations
ranges = [
    decay_range, threshold_range, reset_range, 
    learning_rate_range, initial_weight_range,
    t_pre_range, t_post_range, tau_e_shift_range,
    dw_pos_range, dw_neg_range, 
    min_weight_range, max_weight_range,
    dopamine_correct_range, dopamine_wrong_range,
]

# Printing the total number of configurations
total_configurations = math.prod(map(len, ranges))
print(f"Number of configurations: ", total_configurations)

Number of configurations:  10368


**Logging network activity**

In [70]:
# Initialize history lists
tuning_results = []
epoch_acc = []
num_correct = 0

**Run hyperparameter tuning**

In [ ]:
for config in tqdm(itertools.product(*ranges), total=total_configurations):
    # Unpack configuration hyperparameters
    (decay, threshold, reset, 
     learning_rate, initial_weight, 
     t_pre, t_post, tau_e_shift, 
     dw_pos, dw_neg, 
     min_weight, max_weight, 
     dopamine_correct, dopamine_wrong) = config

    # Setting up parameter configuration dictionaries for neurons and synapses
    neuron_params = {"decay":decay, "threshold":threshold, "reset":reset}
    synapse_params = {"learning_rate":learning_rate, "w_init":initial_weight, 
                      "t_pre":t_pre, "t_post":t_post, "tau_e_shift":tau_e_shift, 
                      "dw_pos":dw_pos, "dw_neg":dw_neg, 
                      "w_min":min_weight, "w_max":max_weight}
    
    # Initialize network with current configuration
    SNN = SNNLayer(n_inputs=n_inputs, n_outputs=n_outputs, synapse_params=synapse_params, neuron_params=neuron_params)

    # Reset acc hist
    epoch_acc = []

    # Repeat test for n epochs
    for n in range(n_epochs):
        # Reset neurons every epoch
        SNN.reset_state()
        # Reset the accuracy list and num_correct every epoch
        num_correct = 0


        # Iterate through spikes in each timestep
        for current_spikes, correct_output in zip(input_spikes, correct_outputs):

            # Forward pass with current timestep
            output_spikes = SNN.forward(input_spikes=current_spikes)

            # Find the "winning" neuron of current timestep
            winner_idx = SNN.winner_takes_all(output_spikes=output_spikes)

            # Check correct
            if winner_idx == correct_output:
                dopamine = dopamine_correct

                # Log number of correct outputs
                num_correct += 1
  
            else:
                dopamine = dopamine_wrong

            # Apply reward
            SNN.apply_reward(dopamine=dopamine, winner_idx=winner_idx)



        epoch_acc.append(num_correct / len(input_spikes))

            
    # Log configuration dictionaries
    tuning_results.append(neuron_params | synapse_params | {"dopamine_correct" : dopamine_correct, "dopamine_wrong" : dopamine_wrong, "mean_acc" : np.mean(epoch_acc)})

100%|██████████| 10368/10368 [18:07<00:00,  9.53it/s]


In [72]:
df_tuning_results = pd.DataFrame(tuning_results)
df_tuning_results.to_csv("CSV_results/SNN_hyperparameter_Results.csv", index=False)
print(f"\nBest config:\n{df_tuning_results.loc[df_tuning_results['mean_acc'].idxmax()]}")


Best config:
decay                  0.75
threshold               2.0
reset                   0.0
learning_rate         0.125
w_init                 None
t_pre                     2
t_post                    2
tau_e_shift               3
dw_pos                0.125
dw_neg               0.0625
w_min               0.03125
w_max                   1.0
dopamine_correct        1.0
dopamine_wrong         -2.0
mean_acc             0.8496
Name: 8762, dtype: object


In [73]:
# Print final weights
import numpy as np
weights = SNN.get_weights()
print("Final weights (rows=output neurons, cols=inputs):")
print(np.round(weights, 3))

# Run one final pass to see per-sample decisions
print("\nPer-sample results:")
for i, (spikes, correct) in enumerate(zip(input_spikes, correct_outputs)):
    out = SNN.forward(input_spikes=spikes)
    winner = SNN.winner_takes_all(out)
    mems = [round(n.mem, 3) for n in SNN.neurons]
    print(f"  Sample {i}: winner={winner} correct={correct} {'✓' if winner==correct else '✗'} spikes={out} mems={mems}")

Final weights (rows=output neurons, cols=inputs):
[[0.031 0.032 0.2   0.031 0.031 0.237 0.108 0.138 0.031 0.091 0.221 0.357
  0.422 0.317 0.209 0.257]
 [0.697 0.041 0.443 0.555 0.551 0.046 0.084 0.435 0.367 0.951 0.181 0.942
  0.991 0.117 0.046 1.   ]
 [0.031 0.031 0.031 0.031 0.031 0.044 0.219 0.031 0.646 0.032 0.377 0.101
  0.06  0.453 0.507 0.38 ]]

Per-sample results:
  Sample 0: winner=2 correct=1 ✗ spikes=[0, 0, 0] mems=[np.float64(0.358), 0.0, np.float64(1.145)]
  Sample 1: winner=2 correct=1 ✗ spikes=[0, 0, 0] mems=[np.float64(0.108), 0.0, np.float64(1.053)]
  Sample 2: winner=2 correct=1 ✗ spikes=[0, 0, 0] mems=[0.0, np.float64(0.719), np.float64(0.773)]
  Sample 3: winner=1 correct=1 ✓ spikes=[0, 0, 0] mems=[0.0, np.float64(1.403), np.float64(0.462)]
  Sample 4: winner=1 correct=1 ✓ spikes=[0, 0, 0] mems=[0.0, np.float64(2.211), np.float64(0.402)]
  Sample 5: winner=1 correct=1 ✓ spikes=[0, 0, 0] mems=[0.0, np.float64(2.464), np.float64(0.31)]
  Sample 6: winner=1 correct=1 ✓

In [ ]:
"""
Interactive SNN Frame-by-Frame Visualizer
=========================================
Paste this entire cell into your Jupyter notebook.
Requires: mem_hist, spike_hist, target_hist, weight_hist, input_spikes, correct_outputs
already populated from your training loop.

Install if needed: pip install ipywidgets matplotlib
"""

import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np


def create_snn_visualizer(mem_hist, spike_hist, target_hist, weight_hist,
                          input_spikes, correct_outputs,
                          threshold=3, samples_per_epoch=50,
                          input_labels=None, output_labels=None):
    """
    Interactive frame-by-frame SNN visualizer.
    
    Args:
        All history lists from training loop.
        input_labels: Optional list of names for input neurons.
        output_labels: Optional list of names for output neurons.
    """
    total_steps = len(mem_hist)
    n_epochs = total_steps // samples_per_epoch
    num_inputs = len(input_spikes[0])
    num_outputs = len(mem_hist[0])

    if input_labels is None:
        input_labels = [f'KP{i}' for i in range(12)] + ['Dist'] + ['ObjL', 'ObjC', 'ObjR']
        if num_inputs != 16:
            input_labels = [f'In{i}' for i in range(num_inputs)]

    if output_labels is None:
        output_labels = ['Left (0)', 'Forward (1)', 'Right (2)']
        if num_outputs != 3:
            output_labels = [f'Out{i}' for i in range(num_outputs)]

    action_map = {0: '← LEFT', 1: '↑ FORWARD', 2: '→ RIGHT'}
    action_colors = {0: '#1f77b4', 1: '#ff7f0e', 2: '#2ca02c'}
    neuron_colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

    # --- Output area ---
    out = widgets.Output()

    def render_frame(step):
        with out:
            clear_output(wait=True)

            epoch = step // samples_per_epoch
            frame = step % samples_per_epoch
            mems = mem_hist[step]
            spks = spike_hist[step]
            target = target_hist[step]
            weights = weight_hist[step] if weight_hist else None
            inp = input_spikes[frame]

            # Determine winner
            spiking = [j for j, s in enumerate(spks) if s > 0]
            if len(spiking) == 1:
                winner = spiking[0]
            elif len(spiking) > 1:
                winner = spiking[0]
            else:
                winner = int(np.argmax(mems))

            correct = winner == target

            # --- Figure ---
            fig = plt.figure(figsize=(18, 10), facecolor='#0d1117')
            gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.5, wspace=0.35,
                                   height_ratios=[1.2, 1.5, 1.2])

            # === Header info ===
            ax_header = fig.add_subplot(gs[0, :])
            ax_header.set_facecolor('#0d1117')
            ax_header.axis('off')

            result_color = '#4ade80' if correct else '#f87171'
            result_text = '✓ CORRECT' if correct else '✗ WRONG'

            header_text = (
                f"Epoch {epoch}  |  Frame {frame}  |  Step {step}/{total_steps-1}\n"
                f"Target: {action_map.get(target, str(target))}  |  "
                f"Winner: {action_map.get(winner, str(winner))}  |  "
            )
            ax_header.text(0.02, 0.7, header_text, transform=ax_header.transAxes,
                          fontsize=16, color='#c9d1d9', fontfamily='monospace',
                          verticalalignment='top')
            ax_header.text(0.72, 0.7, result_text, transform=ax_header.transAxes,
                          fontsize=22, color=result_color, fontweight='bold',
                          fontfamily='monospace', verticalalignment='top')

            # === Input spikes (left panel) ===
            ax_inp = fig.add_subplot(gs[1, 0])
            ax_inp.set_facecolor('#161b22')
            y_pos = np.arange(num_inputs)
            bar_colors = ['#58a6ff' if v > 0 else '#21262d' for v in inp]
            ax_inp.barh(y_pos, inp, color=bar_colors, height=0.7, edgecolor='#30363d')
            ax_inp.set_yticks(y_pos)
            ax_inp.set_yticklabels(input_labels, fontsize=9, color='#c9d1d9', fontfamily='monospace')
            ax_inp.set_xlim(-0.1, 1.5)
            ax_inp.set_title('Input Spikes', fontsize=13, color='#c9d1d9',
                            fontweight='bold', pad=10)
            ax_inp.tick_params(colors='#484f58', labelsize=9)
            ax_inp.spines[:].set_color('#30363d')
            ax_inp.invert_yaxis()

            # === Membrane potentials (center panel) ===
            ax_mem = fig.add_subplot(gs[1, 1])
            ax_mem.set_facecolor('#161b22')
            y_pos_out = np.arange(num_outputs)
            bar_colors_mem = []
            for j in range(num_outputs):
                if spks[j]:
                    bar_colors_mem.append('#f87171')  # fired = red
                elif j == winner:
                    bar_colors_mem.append(neuron_colors[j % len(neuron_colors)])
                else:
                    bar_colors_mem.append('#484f58')

            bars = ax_mem.barh(y_pos_out, mems, color=bar_colors_mem, height=0.6,
                              edgecolor='#30363d', linewidth=1.5)
            ax_mem.axvline(x=threshold, color='#f0883e', linestyle='--',
                          linewidth=2, alpha=0.8, label=f'θ = {threshold}')

            # Labels on bars
            for j, (m, s) in enumerate(zip(mems, spks)):
                label = f'{m:.2f} ⚡' if s else f'{m:.2f}'
                ax_mem.text(max(m, 0) + 0.05, j, label, va='center',
                           fontsize=11, color='#c9d1d9', fontfamily='monospace')

            ax_mem.set_yticks(y_pos_out)
            ax_mem.set_yticklabels(output_labels, fontsize=11, color='#c9d1d9', fontfamily='monospace')
            ax_mem.set_xlim(0, threshold * 1.4)
            ax_mem.set_title('Membrane Potentials', fontsize=13, color='#c9d1d9',
                            fontweight='bold', pad=10)
            ax_mem.legend(fontsize=10, loc='lower right', facecolor='#161b22',
                         edgecolor='#30363d', labelcolor='#c9d1d9')
            ax_mem.tick_params(colors='#484f58', labelsize=10)
            ax_mem.spines[:].set_color('#30363d')
            ax_mem.invert_yaxis()

            # === Weight heatmap (right panel) ===
            if weights is not None:
                ax_w = fig.add_subplot(gs[1, 2])
                ax_w.set_facecolor('#161b22')
                im = ax_w.imshow(weights, aspect='auto', cmap='inferno',
                                vmin=0, vmax=1.0, interpolation='nearest')
                ax_w.set_yticks(range(num_outputs))
                ax_w.set_yticklabels(output_labels, fontsize=10, color='#c9d1d9', fontfamily='monospace')
                ax_w.set_xticks(range(num_inputs))
                ax_w.set_xticklabels(input_labels, fontsize=7, color='#c9d1d9',
                                    rotation=45, ha='right', fontfamily='monospace')
                ax_w.set_title('Weight Matrix', fontsize=13, color='#c9d1d9',
                              fontweight='bold', pad=10)
                ax_w.tick_params(colors='#484f58')
                ax_w.spines[:].set_color('#30363d')
                cbar = plt.colorbar(im, ax=ax_w, fraction=0.046, pad=0.04)
                cbar.ax.tick_params(colors='#c9d1d9', labelsize=9)

            # === Membrane history (bottom panel, trailing window) ===
            ax_trail = fig.add_subplot(gs[2, :])
            ax_trail.set_facecolor('#161b22')
            window = min(samples_per_epoch * 2, step + 1)
            start_t = max(0, step - window + 1)

            for j in range(num_outputs):
                trail_mems = [mem_hist[t][j] for t in range(start_t, step + 1)]
                trail_spks = [spike_hist[t][j] for t in range(start_t, step + 1)]
                x_range = range(start_t, step + 1)
                ax_trail.plot(x_range, trail_mems, color=neuron_colors[j % len(neuron_colors)],
                             linewidth=1.2, alpha=0.85, label=output_labels[j])

                sp_times = [t for t, s in zip(x_range, trail_spks) if s > 0]
                if sp_times:
                    ax_trail.scatter(sp_times, [threshold] * len(sp_times),
                                   color=neuron_colors[j % len(neuron_colors)],
                                   marker='v', s=35, zorder=5, alpha=0.9)

            # Target shading in trail
            for t in range(start_t, step + 1):
                tgt = target_hist[t]
                ax_trail.axvspan(t, t + 1,
                                color=neuron_colors[tgt % len(neuron_colors)], alpha=0.06)

            ax_trail.axhline(y=threshold, color='#f0883e', linestyle='--', linewidth=1.5, alpha=0.6)
            ax_trail.axvline(x=step, color='#f87171', linestyle='-', linewidth=2, alpha=0.7)
            ax_trail.set_title('Membrane History (trailing window)', fontsize=13,
                              color='#c9d1d9', fontweight='bold', pad=10)
            ax_trail.set_xlabel('Timestep', fontsize=11, color='#c9d1d9')
            ax_trail.set_ylabel('Membrane', fontsize=11, color='#c9d1d9')
            ax_trail.legend(fontsize=9, loc='upper left', facecolor='#161b22',
                           edgecolor='#30363d', labelcolor='#c9d1d9', ncol=num_outputs)
            ax_trail.tick_params(colors='#484f58', labelsize=10)
            ax_trail.spines[:].set_color('#30363d')

            plt.show()

    # --- Controls ---
    step_slider = widgets.IntSlider(
        value=0, min=0, max=total_steps - 1, step=1,
        description='Step:', continuous_update=False,
        layout=widgets.Layout(width='70%'),
        style={'description_width': '50px'}
    )

    epoch_slider = widgets.IntSlider(
        value=0, min=0, max=n_epochs - 1, step=1,
        description='Epoch:', continuous_update=False,
        layout=widgets.Layout(width='40%'),
        style={'description_width': '50px'}
    )

    frame_slider = widgets.IntSlider(
        value=0, min=0, max=samples_per_epoch - 1, step=1,
        description='Frame:', continuous_update=False,
        layout=widgets.Layout(width='40%'),
        style={'description_width': '50px'}
    )

    play_btn = widgets.Play(
        value=0, min=0, max=total_steps - 1, step=1,
        interval=300, description='▶',
        layout=widgets.Layout(width='60px')
    )

    speed_dropdown = widgets.Dropdown(
        options=[('Slow (500ms)', 500), ('Normal (300ms)', 300),
                 ('Fast (100ms)', 100), ('Very Fast (50ms)', 50)],
        value=300, description='Speed:',
        layout=widgets.Layout(width='200px'),
        style={'description_width': '50px'}
    )

    # Link play button to step slider
    widgets.jslink((play_btn, 'value'), (step_slider, 'value'))

    def on_speed_change(change):
        play_btn.interval = change['new']
    speed_dropdown.observe(on_speed_change, names='value')

    # Sync epoch/frame to step
    _updating = [False]

    def on_step_change(change):
        if _updating[0]:
            return
        _updating[0] = True
        s = change['new']
        epoch_slider.value = s // samples_per_epoch
        frame_slider.value = s % samples_per_epoch
        render_frame(s)
        _updating[0] = False

    def on_epoch_frame_change(change):
        if _updating[0]:
            return
        _updating[0] = True
        s = epoch_slider.value * samples_per_epoch + frame_slider.value
        step_slider.value = min(s, total_steps - 1)
        render_frame(min(s, total_steps - 1))
        _updating[0] = False

    step_slider.observe(on_step_change, names='value')
    epoch_slider.observe(on_epoch_frame_change, names='value')
    frame_slider.observe(on_epoch_frame_change, names='value')

    # Layout
    controls_top = widgets.HBox([play_btn, step_slider, speed_dropdown])
    controls_bottom = widgets.HBox([epoch_slider, frame_slider])
    ui = widgets.VBox([controls_top, controls_bottom, out])

    render_frame(0)
    display(ui)


# === RUN THIS ===
create_snn_visualizer(
    mem_hist=mem_hist,
    spike_hist=spike_hist,
    target_hist=target_hist,
    weight_hist=weight_hist,
    input_spikes=input_spikes,
    correct_outputs=correct_outputs,
    threshold=2.5,
    samples_per_epoch=len(input_spikes)
)